<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇷🇴 Romania Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    Source: github.com/vasile/data.gov.ro-gtfs-exporter · Provider: CFR Călători · Format: GTFS only · NeTEx not available
  </p>
</div>

## Data Source

**GTFS — CFR Călători Romania (community conversion)**
- Source: GitHub repository by Vasile Coțovanu
- Dataset page: https://github.com/vasile/data.gov.ro-gtfs-exporter/tree/master/gtfs-out
- License: MIT License
- The GTFS files are not an official feed — they are converted from the
  official CFR Călători XML timetables published on `data.gov.ro` using
  a Ruby script maintained by the community
- Despite being unofficial, the conversion produced a clean and usable
  GTFS dataset with 1,695 stops, 1,023 routes and 203 service IDs

**NeTEx — not available**

No NeTEx dataset could be found for Romania through any official source.
Romania does not publish NeTEx data publicly.

# Part 1: GTFS Exploration

All third-party and standard-library imports used in this notebook are consolidated here.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd
from collections import defaultdict

In [2]:
# Set path

data_dir = Path("data")

gtfs_zip_path = data_dir / "data.gov.ro-gtfs-exporter-master.zip"

print("GTFS exists:", gtfs_zip_path.exists(), gtfs_zip_path)

GTFS exists: True data/data.gov.ro-gtfs-exporter-master.zip


In [3]:
# Inspect GTFS ZIP contents
with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = pd.DataFrame([
        {
            "name":    info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(gtfs_files.sort_values("size_mb", ascending=False))

,name,size_mb
19,data.gov.ro-gtfs-exporter-master/gtfs-out/stop...,0.90
7,data.gov.ro-gtfs-exporter-master/cfr.webgis.ro...,0.57
17,data.gov.ro-gtfs-exporter-master/gtfs-out/cale...,0.27
12,data.gov.ro-gtfs-exporter-master/docs/Romanian...,0.19
11,data.gov.ro-gtfs-exporter-master/docs/Romanian...,0.15
20,data.gov.ro-gtfs-exporter-master/gtfs-out/stop...,0.06
18,data.gov.ro-gtfs-exporter-master/gtfs-out/rout...,0.04
21,data.gov.ro-gtfs-exporter-master/gtfs-out/trip...,0.03
22,data.gov.ro-gtfs-exporter-master/gtfs.rb,0.02
16,data.gov.ro-gtfs-exporter-master/gtfs-out/cale...,0.01


## Load GTFS files for Romania

The ready-made GTFS files are stored inside the `gtfs-out` folder within
the downloaded ZIP archive. Since the files are not in a standard GTFS ZIP
but nested inside a repository ZIP, they are read directly from the archive
using the full internal path.

In [4]:
def read_gtfs_from_zip(zip_path, filename):
    with zipfile.ZipFile(zip_path, "r") as z:
        # The files are inside gtfs-out subfolder within the ZIP
        with z.open(f"data.gov.ro-gtfs-exporter-master/gtfs-out/{filename}") as f:
            return pd.read_csv(f, dtype=str, low_memory=False)

stops          = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes         = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips          = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
calendar       = read_gtfs_from_zip(gtfs_zip_path, "calendar.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")

print("Stops:", len(stops))
print("Routes:", len(routes))
print("Trips:", len(trips))
print("Calendar:", len(calendar))
print("Calendar dates:", len(calendar_dates))

Stops: 1695
Routes: 1023
Trips: 2103
Calendar: 203
Calendar dates: 19864


In [5]:
# Inspect GTFS file sizes and structure

print("Files in gtfs-out folder:")
with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = [
        {"name": f.filename, "size_mb": round(f.file_size / 1024**2, 2)}
        for f in z.infolist()
        if "gtfs-out" in f.filename and f.filename.endswith(".txt")
    ]
display(pd.DataFrame(gtfs_files).sort_values("size_mb", ascending=False))

# Basic inspection
print("\nSample stops:")
display(stops.head(5))
print("\nSample routes:")
display(routes.head(5))
print("\nCalendar columns:", calendar.columns.tolist())
print("Calendar dates columns:", calendar_dates.columns.tolist())

Files in gtfs-out folder:


,name,size_mb
4,data.gov.ro-gtfs-exporter-master/gtfs-out/stop...,0.90
2,data.gov.ro-gtfs-exporter-master/gtfs-out/cale...,0.27
5,data.gov.ro-gtfs-exporter-master/gtfs-out/stop...,0.06
3,data.gov.ro-gtfs-exporter-master/gtfs-out/rout...,0.04
6,data.gov.ro-gtfs-exporter-master/gtfs-out/trip...,0.03
1,data.gov.ro-gtfs-exporter-master/gtfs-out/cale...,0.01
0,data.gov.ro-gtfs-exporter-master/gtfs-out/agen...,0.00



Sample stops:


,stop_id,stop_name,stop_lat,stop_lon
0,123,Vidin Patnicheska,43.974565,22.851949
1,936,Ungheni Prut Fr.,47.199948,27.786977
2,09410,Nyirabrany,47.523867,22.016129
3,09422,Biharkeresztes,47.135644,21.721945
4,09446,Kotegyan,46.758621,21.478958



Sample routes:


,route_id,agency_id,route_short_name,route_long_name,route_type
0,1,228389,NaN,Măneciu - Ploieşti Sud,106
1,2,228389,NaN,Bârlad - Chiraftei h.,200
2,3,228389,NaN,Chiraftei h. - Bârlad,200
3,4,228389,NaN,Aghireş Hm. - Cluj Napoca,200
4,5,228389,NaN,Nehoiaşu Hm. - Buzău,106



Calendar columns: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
Calendar dates columns: ['service_id', 'date', 'exception_type']


In [6]:
# Inspect each GTFS table 
gtfs_file_summary = pd.DataFrame({
    "file": ["stops.txt", "routes.txt", "trips.txt", "calendar.txt", "calendar_dates.txt"],
    "rows": [len(stops), len(routes), len(trips), len(calendar), len(calendar_dates)]
})
display(gtfs_file_summary)

print("\nSample stops:")
display(stops.head(5))

print("\nSample routes:")
display(routes[["route_id", "route_short_name", "route_long_name", "route_type"]].head(5))

print("\nSample calendar:")
display(calendar.head(5))

print("\nSample calendar_dates:")
display(calendar_dates.head(5))

,file,rows
0,stops.txt,1695
1,routes.txt,1023
2,trips.txt,2103
3,calendar.txt,203
4,calendar_dates.txt,19864



Sample stops:


,stop_id,stop_name,stop_lat,stop_lon
0,123,Vidin Patnicheska,43.974565,22.851949
1,936,Ungheni Prut Fr.,47.199948,27.786977
2,09410,Nyirabrany,47.523867,22.016129
3,09422,Biharkeresztes,47.135644,21.721945
4,09446,Kotegyan,46.758621,21.478958



Sample routes:


,route_id,route_short_name,route_long_name,route_type
0,1,NaN,Măneciu - Ploieşti Sud,106
1,2,NaN,Bârlad - Chiraftei h.,200
2,3,NaN,Chiraftei h. - Bârlad,200
3,4,NaN,Aghireş Hm. - Cluj Napoca,200
4,5,NaN,Nehoiaşu Hm. - Buzău,106



Sample calendar:


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1,1,1,1,1,1,1,1,20251214,20261212
1,2,1,1,1,1,1,1,1,20251214,20261212
2,3,1,1,1,1,1,1,1,20251214,20261212
3,4,1,1,1,1,1,1,1,20251214,20261212
4,5,1,1,1,1,1,1,1,20251214,20261212



Sample calendar_dates:


,service_id,date,exception_type
0,2,20260101,2
1,2,20260102,2
2,2,20260103,2
3,2,20260104,2
4,2,20260105,2


## Comment

A first look at the Romanian GTFS tables reveals several interesting findings:

- **1,695 stops**, all with names and coordinates, only 4 columns present
  (no `location_type`, `parent_station` or `stop_code`)
- **1,023 routes**, all missing `route_short_name`, using `route_long_name`
  only with origin-destination format like "Măneciu - Ploiești Sud". Route
  types include 106 (regional rail) and 200 (unknown/non-standard)
- **2,103 trips** and **203 service IDs** in `calendar.txt`
- **Calendar** uses both `calendar.txt` (weekly patterns) and
  `calendar_dates.txt` (19,864 exception rows, all type 2 — removed dates)
- The calendar validity window runs from **2025-12-14 to 2026-12-12** — a
  full year
- Some stops have non-Romanian names like for example "Vidin Patnicheska", "Nyirabrany",
  "Biharkeresztes", "Kotegyan", suggesting the feed includes cross-border
  stations in Bulgaria and Hungary
- All service IDs in `calendar.txt` run every day of the week (all 7
  weekday flags = 1), with exceptions removed via `calendar_dates.txt`

## Prepare GTFS station table

Since the Romanian GTFS feed has no `location_type` or `parent_station`
columns, all 1,695 stops are used directly as the station-level table,
the same approach used for Italy, Finland and Sweden.

In [7]:
# Prepare GTFS station table for Romania
gtfs_stations = stops.copy()
gtfs_stations = gtfs_stations.rename(columns={
    "stop_id":   "gtfs_stop_id",
    "stop_name": "gtfs_stop_name",
    "stop_lat":  "gtfs_lat",
    "stop_lon":  "gtfs_lon"
})

gtfs_station_quality = pd.DataFrame({
    "metric": [
        "total station rows",
        "rows with name",
        "rows missing name",
        "rows with coordinates",
        "rows missing coordinates",
        "unique station IDs",
        "duplicate station IDs"
    ],
    "value": [
        len(gtfs_stations),
        gtfs_stations["gtfs_stop_name"].notna().sum(),
        gtfs_stations["gtfs_stop_name"].isna().sum(),
        gtfs_stations[["gtfs_lat", "gtfs_lon"]].notna().all(axis=1).sum(),
        gtfs_stations[["gtfs_lat", "gtfs_lon"]].isna().any(axis=1).sum(),
        gtfs_stations["gtfs_stop_id"].nunique(),
        gtfs_stations["gtfs_stop_id"].duplicated().sum()
    ]
})
display(gtfs_station_quality)
display(gtfs_stations.head(10))

,metric,value
0,total station rows,1695
1,rows with name,1695
2,rows missing name,0
3,rows with coordinates,1695
4,rows missing coordinates,0
5,unique station IDs,1695
6,duplicate station IDs,0


,gtfs_stop_id,gtfs_stop_name,gtfs_lat,gtfs_lon
0,123,Vidin Patnicheska,43.974565,22.851949
1,936,Ungheni Prut Fr.,47.199948,27.786977
2,09410,Nyirabrany,47.523867,22.016129
3,09422,Biharkeresztes,47.135644,21.721945
4,09446,Kotegyan,46.758621,21.478958
5,09461,Lokoshaza,46.432193,21.238042
6,09711,Ruse,43.857121,25.976744
7,10017,Bucureşti Nord Gr.A,44.446652,26.073465
8,10079,Bucureşti Basarab h.,44.458753,26.055381
9,10081,Carpaţi h. (900),44.468198,26.040908


## Comment

The GTFS station table is clean and complete:

- All **1,695 stops** have names and coordinates with no missing values,
  all IDs unique
- No `location_type` or `parent_station`, a flat structure as expected
- The sample confirms cross-border coverage because of stops
  like Vidin Patnicheska (Bulgaria),Nyirabrany, Biharkeresztes, Kotegyan, 
  Lokoshaza (Hungary), Ruse (Bulgaria) all appear alongside Romanian stations 
  like București Nord
- Coordinates look correct, latitudes between 43° and 48°N, longitudes
  between 22° and 30°E, consistent with Romania and neighbouring countries

## Prepare GTFS line table

Since all routes are missing `route_short_name`, the `route_long_name` is
used as the line label, same approach as Italy and Spain. The route type
distribution is also checked since the sample showed unusual codes like 200.

In [8]:
# Prepare GTFS line table for Romania
gtfs_lines = routes.copy()
gtfs_lines["gtfs_line_label"] = gtfs_lines["route_short_name"].fillna(
    gtfs_lines["route_long_name"]
)
gtfs_lines = gtfs_lines.rename(columns={
    "route_id":   "gtfs_route_id",
    "route_type": "gtfs_route_type"
})[["gtfs_route_id", "gtfs_line_label", "gtfs_route_type"]]

gtfs_line_quality = pd.DataFrame({
    "metric": [
        "total line rows",
        "rows with line label",
        "rows missing line label",
        "unique line labels",
        "duplicate line labels"
    ],
    "value": [
        len(gtfs_lines),
        gtfs_lines["gtfs_line_label"].notna().sum(),
        gtfs_lines["gtfs_line_label"].isna().sum(),
        gtfs_lines["gtfs_line_label"].nunique(),
        gtfs_lines["gtfs_line_label"].duplicated().sum()
    ]
})
display(gtfs_line_quality)

# Route type distribution
route_type_names = {
    2:   "Rail",
    100: "Railway Service",
    101: "High Speed Rail",
    102: "Long Distance Rail",
    106: "Regional Rail",
    109: "Suburban Rail",
    200: "Coach / Long Distance Bus"
}
routes["route_type_int"] = routes["route_type"].astype(int)
routes["route_type_name"] = routes["route_type_int"].map(route_type_names)

print("Route type distribution:")
display(
    routes.groupby(["route_type_int", "route_type_name"])
    .size()
    .reset_index(name="n_routes")
    .sort_values("n_routes", ascending=False)
)

print("Sample lines:")
display(gtfs_lines.head(10))

,metric,value
0,total line rows,1023
1,rows with line label,1023
2,rows missing line label,0
3,unique line labels,613
4,duplicate line labels,410


Route type distribution:


,route_type_int,route_type_name,n_routes
1,106,Regional Rail,774
2,200,Coach / Long Distance Bus,40
0,102,Long Distance Rail,30


Sample lines:


,gtfs_route_id,gtfs_line_label,gtfs_route_type
0,1,Măneciu - Ploieşti Sud,106
1,2,Bârlad - Chiraftei h.,200
2,3,Chiraftei h. - Bârlad,200
3,4,Aghireş Hm. - Cluj Napoca,200
4,5,Nehoiaşu Hm. - Buzău,106
5,6,Galaţi - Târgu Bujor,106
6,7,Ploieşti Sud - Slănic Hm.,106
7,8,Titan Sud Hm. - Olteniţa,106
8,9,Aeroport H. Coanda T1 - Bucureşti Nord Gr.A,106
9,10,Chiraftei h. - Galaţi,106


## Comment

The Romanian GTFS line table reveals a mixed transport network:

- **1,023 routes** in total, all with a line label from `route_long_name`
- **613 unique labels**; 410 duplicate labels exist because the same
  origin-destination name is used for both directions of travel (e.g.
  "Bârlad - Chiraftei h." and "Chiraftei h. - Bârlad" are two separate
  routes for the same line)
- Three route types are present:
  - **106 Regional Rail** - 774 routes, the dominant mode
  - **200 Coach / Long Distance Bus** — 40 routes, likely replacement
    bus services
  - **102 Long Distance Rail** — 30 routes
- Labels follow the same origin-destination format as Italy, no short
  public codes are available

## Prepare GTFS calendar

Romania uses both `calendar.txt` and `calendar_dates.txt`. The sample showed
that all services run every day of the week in `calendar.txt`, with specific
dates removed via exception type 2 in `calendar_dates.txt`. Active dates are
rebuilt by applying the weekly pattern and then removing the exception dates.

In [9]:
# Convert dates
calendar["start_date_dt"] = pd.to_datetime(calendar["start_date"], format="%Y%m%d")
calendar["end_date_dt"]   = pd.to_datetime(calendar["end_date"],   format="%Y%m%d")
calendar_dates["date_dt"] = pd.to_datetime(calendar_dates["date"], format="%Y%m%d")
calendar_dates["exception_type"] = calendar_dates["exception_type"].astype(int)

weekday_cols = [
    "monday", "tuesday", "wednesday", "thursday",
    "friday", "saturday", "sunday"
]

feed_start = calendar["start_date_dt"].min()
feed_end   = calendar["end_date_dt"].max()

print(f"Feed window: {feed_start.date()} to {feed_end.date()}")

# Build added and removed dates
calendar["service_id"]       = calendar["service_id"].astype(str)
calendar_dates["service_id"] = calendar_dates["service_id"].astype(str)

added_dates   = defaultdict(set)
removed_dates = defaultdict(set)

for _, row in calendar_dates.iterrows():
    if row["exception_type"] == 1:
        added_dates[row["service_id"]].add(row["date_dt"])
    elif row["exception_type"] == 2:
        removed_dates[row["service_id"]].add(row["date_dt"])

# Rebuild active dates: apply each service's own weekly pattern over its own
# start/end range, then apply the added/removed exception dates on top
gtfs_active_date_rows = []
for _, row in calendar.iterrows():
    sid = row["service_id"]
    all_days = pd.date_range(row["start_date_dt"], row["end_date_dt"], freq="D")
    active_dates = {d for d in all_days if row[weekday_cols[d.weekday()]] == "1"}
    active_dates = (active_dates - removed_dates[sid]) | added_dates[sid]

    for d in sorted(active_dates):
        gtfs_active_date_rows.append({"service_id": sid, "active_date": d})

gtfs_active_dates = pd.DataFrame(gtfs_active_date_rows)
print(f"Active dates built for {gtfs_active_dates['service_id'].nunique()} service IDs.")
print(f"Total active date rows: {len(gtfs_active_dates)}")
display(gtfs_active_dates.head(10))

Feed window: 2025-12-14 to 2026-12-12
Active dates built for 200 service IDs.
Total active date rows: 41184


,service_id,active_date
0,1,2025-12-14
1,1,2025-12-15
2,1,2025-12-16
3,1,2025-12-17
4,1,2025-12-18
5,1,2025-12-19
6,1,2025-12-20
7,1,2025-12-21
8,1,2025-12-22
9,1,2025-12-23


## Comment

The active dates were built successfully:

- **200 service IDs** with active dates , slightly fewer than the 203
  service IDs in `calendar.txt`, meaning 3 service IDs had no active
  dates after removing exceptions
- **41,184 total active date rows** covering the feed window from
  2025-12-14 to 2026-12-12 so a full year
- Service ID 1 runs every day from the start of the feed , consistent
  with the all-days-active weekly pattern seen in `calendar.txt`
- The large number of exception type 2 rows (19,864 removals) suggests
  many services have specific days cancelled, likely for public holidays
  or seasonal schedule changes



## Data quality check

A final quality check on the calendar table to confirm the data
is complete. 

In [10]:
# Calendar quality check
calendar_quality = pd.DataFrame({
    "metric": [
        "total calendar rows",
        "unique service_id values",
        "total calendar_dates rows",
        "exception type 1 rows (added dates)",
        "exception type 2 rows (removed dates)",
        "earliest date",
        "latest date",
        "number of days in feed window"
    ],
    "value": [
        len(calendar),
        calendar["service_id"].nunique(),
        len(calendar_dates),
        (calendar_dates["exception_type"] == 1).sum(),
        (calendar_dates["exception_type"] == 2).sum(),
        calendar_dates["date_dt"].min(),
        calendar_dates["date_dt"].max(),
        (calendar_dates["date_dt"].max() - calendar_dates["date_dt"].min()).days + 1
    ]
})
display(calendar_quality)

,metric,value
0,total calendar rows,203
1,unique service_id values,203
2,total calendar_dates rows,19864
3,exception type 1 rows (added dates),0
4,exception type 2 rows (removed dates),19864
5,earliest date,2025-12-14 00:00:00
6,latest date,2026-12-12 00:00:00
7,number of days in feed window,364


## Comment

**Calendar:**
- 203 unique service IDs in `calendar.txt` — one per service, clean
- All 19,864 `calendar_dates.txt` rows are **exception type 2** (removed
  dates), no added dates at all. This means the entire calendar is
  built from the weekly all-days pattern with specific days removed
- Feed window covers **364 days** from 2025-12-14 to 2026-12-12 
- The high number of removals (19,864 across 203 services) suggests
  many specific days are cancelled , likely public holidays, track
  maintenance periods or seasonal schedule changes


In [11]:
gtfs_summary = pd.DataFrame({
    "component": ["Stops", "Lines", "Calendar"],
    "key finding": [
        "1,695 stops — flat structure, no hierarchy, includes cross-border stops in Bulgaria and Hungary",
        "1,023 routes — no route_short_name, origin-destination labels only, 3 route types (106, 102, 200)",
        "200 active service IDs — all-days weekly pattern with exception type 2 removals, valid 2025-12-14 to 2026-12-12"
    ]
})
pd.set_option("display.max_colwidth", None)
display(gtfs_summary)

,component,key finding
0,Stops,"1,695 stops — flat structure, no hierarchy, includes cross-border stops in Bulgaria and Hungary"
1,Lines,"1,023 routes — no route_short_name, origin-destination labels only, 3 route types (106, 102, 200)"
2,Calendar,"200 active service IDs — all-days weekly pattern with exception type 2 removals, valid 2025-12-14 to 2026-12-12"


## Conclusion: Romania

Like Spain, the GTFS–NeTEx comparison could not be performed for Romania
due to the absence of publicly available NeTEx data.

Romania does not publish an official GTFS feed either. The dataset used
here is a community conversion of the proprietary CFR Călători XML format.
Despite being unofficial, it worked well and produced clean, usable data.

Romania and Spain together show that standardised open transport data is
still missing in several EU member states, which will be discussed in the
overall thesis conclusions.